# ML Assignment 02
### Dataset: House Price Prediction Dataset
### Total Marks: 100

---

## Exam Instructions:
1. প্রথমে নিচের cell এ নিজের **নাম** এবং কোর্সে registration করা **ইমেইল** দিবে
2. Question wise numbering করে Text cell রাখবে এবং এর নিচে Code cell থাকবে, চেষ্টা করবে একটি code cell এ একটি question উত্তর দেওয়ার
3. Google Colab এর মধ্যে কোডগুলো করবে
4. এবং সেই ফাইলটি **'Anyone with the link' & 'View' Access** দিয়ে ফাইলটির Shareable Link টি সাবমিট করবে

---

**Question Dataset Link:** https://www.kaggle.com/datasets/prokshitha/home-value-insights

## Student Information

In [ ]:
# Fill in your information
name = "Al Fattaul Islam"           # Write your full name here
email = "alfattaulislamss@gmail.com"          # Write your registered email here

print(f"Name  : {name}")
print(f"Email : {email}")

Name  : Al Fattaul Islam
Email : alfattaulislamss@gmail.com


---
## Question 1 (10 Marks)

Load the House Price dataset and display:
- Dataset shape
- First 10 rows
- 5 random samples

In [ ]:
# Question 1
import pandas as pd

# Load the House Price dataset and display:
df = pd.read_csv('house_price_regression_dataset.csv')

# Dataset shape
print('Dataset Shape:', df.shape)
print()

# First 10 rows
print('First 10 rows: \n', df.head(10))
print()

# 5 random samples
print('5 random samples: \n', df.sample(5))

Dataset Shape: (1000, 8)

First 10 rows: 
    Square_Footage  Num_Bedrooms  Num_Bathrooms  Year_Built  Lot_Size  \
0            1360             2              1        1981  0.599637   
1            4272             3              3        2016  4.753014   
2            3592             1              2        2016  3.634823   
3             966             1              2        1977  2.730667   
4            4926             2              1        1993  4.699073   
5            3944             5              3        1990  2.475930   
6            3671             1              2        2012  4.911960   
7            3419             1              1        1972  2.805281   
8             630             3              3        1997  1.014286   
9            2185             4              2        1981  3.941604   

   Garage_Size  Neighborhood_Quality   House_Price  
0            0                     5  2.623829e+05  
1            1                     6  9.852609e+05  
2    

---
## Question 2 (10 Marks)

Handle missing values and perform feature engineering:
- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.

In [ ]:
#Question 2
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

# Drop columns with more than 50% missing values
thresh = len(df) * 0.5
clean_df = df.dropna(thresh = thresh, axis = 1)

# Impute missing numerical values using SimpleImputer with mean strategy
num_cols = clean_df.select_dtypes(include = ['int64', 'float64']).columns
num_imputer = SimpleImputer(strategy='mean')
clean_df[num_cols] = num_imputer.fit_transform(clean_df[num_cols])

# Impute missing categorical values using most frequent strategy
cat_cols = clean_df.select_dtypes(include = ['object']).columns
if len(cat_cols) > 0:
  cat_imputer = SimpleImputer(strategy='most_frequent')
  clean_df[cat_cols] = cat_imputer.fit_transform(clean_df[cat_cols])

# Perform train-test split with test_size=0.2 and random_state=42
X = clean_df.drop('House_Price', axis = 1)
y = clean_df['House_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Display the shape of final train and test sets.
print('Shape of X_train:', X_train.shape)
print('Shape of X_test:', X_test.shape)

Shape of X_train: (800, 7)
Shape of X_test: (200, 7)


---
## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):
- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).

In [ ]:
#Question 3
# Use Square_Footage as feature (X) and House_Price as target (y).
X = clean_df['Square_Footage']
y = clean_df['House_Price']

X_train, X_test_nrm, y_train, y_test_nrm = train_test_split(X, y, test_size=0.2, random_state=42)

# normalize
X_mean = X_train.mean()
X_std = X_train.std()
y_mean = y_train.mean()
y_std = y_train.std()

X_train_norm = (X_train - X_mean) / X_std
X_test_norm = (X_test_nrm - X_mean) / X_std
y_train_norm = (y_train - y_mean) / y_std

# Compute slope (m) and intercept (c) using the Batch Gradient Descent
def compute_cost(X, y, w, b):
  prediction = w * X + b
  error = prediction - y
  cost = np.sum(error ** 2) / (2 * len(X))
  return cost

def calculate_gradient(X, y, w, b):
  prediction = w * X + b
  error = prediction - y
  dj_dw = np.sum(error * X) / len(X)
  dj_db = np.sum(error) / len(X)
  return dj_dw, dj_db

def gradient_descent(X, y, w_input, b_input, max_iter, alpha = 0.01):
  w = w_input
  b = b_input
  cost_memo = []
  iteration = []

  for i in range(max_iter):
    dj_dw, dj_db = calculate_gradient(X, y, w, b)

    w = w - alpha * dj_dw
    b = b - alpha * dj_db

    cost = compute_cost(X, y, w, b)
    cost_memo.append(cost)
    iteration.append(i)

  return w, b, cost_memo, iteration

w_final, b_final, cost_memo, iter_list = gradient_descent(X_train_norm, y_train_norm, w_input = 0, b_input = 0, max_iter = 1000, alpha = 0.01)

# Predict values for the test set
predict = w_final * X_test_norm + b_final
y_pred = predict * y_std + y_mean
print('Predicted Value: \n', y_pred[:6])
print()

# Print the learned m and c values
m_val = w_final * (y_std / X_std)
c_val = y_mean - m_val * X_mean
print('Slope (m):', round(m_val, 4))
print('Intercept (c):', round(c_val, 4))

Predicted Value: 
 521    8.588527e+05
737    5.175199e+05
740    9.984342e+05
660    1.043357e+06
411    7.854521e+05
678    7.734192e+05
Name: Square_Footage, dtype: float64

Slope (m): 200.5481
Intercept (c): 54253.7317


---
## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:
- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`



In [ ]:
#Question 4
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler,OneHotEncoder

# StandardScaler on numerical columns: Square_Footage, Num_Bedrooms, Num_Bathrooms
# OneHotEncoder on categorical column: Neighborhood_Quality
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']),
        ('cat', OneHotEncoder(handle_unknown = 'ignore'), ['Neighborhood_Quality'])
    ]
)

preprocessor

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['Square_Footage', 'Num_Bedrooms',
                                  'Num_Bathrooms']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['Neighborhood_Quality'])])

## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:
- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [ ]:
# Question 5
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

X = clean_df.drop('House_Price', axis = 1)
y = clean_df['House_Price']

X_train, X_test , y_train, y_test = train_test_split(X,y, test_size = 0.20 , random_state = 42)

# SGDRegressor as the final estimator
# The ColumnTransformer
sgd_pipeline = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', SGDRegressor(
            loss = 'squared_error',
            penalty = 'l2',
            alpha = 0.0001,
            max_iter = 1000,
            learning_rate = 'constant',
            eta0 = 0.01,
            random_state = 42
        ))
    ]
)


# Train the pipeline and evaluate using RMSE and R² score
# Train the pipeline
sgd_train = sgd_pipeline.fit(X_train, y_train)

# evaluate using RMSE and R² score
def evaluate(model, name):
  train_pred = model.predict(X_train)
  test_pred = model.predict(X_test)

  train_RMSE = np.sqrt(mean_squared_error(y_train, train_pred))
  test_RMSE = np.sqrt(mean_squared_error(y_test, test_pred))
  train_r2 = r2_score(y_train, train_pred)
  test_r2 = r2_score(y_test, test_pred)

  return {
      'model'             : name,
      'Train RMSE'        : round(train_RMSE, 4),
      'Test RMSE'         : round(test_RMSE, 4),
      'Train R2'          : round(train_r2, 4),
      'Test R2'           : round(test_r2, 4)
  }

# SGDRegressor as the final estimator
sgd_result = evaluate(sgd_pipeline, 'SGD Regressor')
print('SGD Output:', sgd_result)
print()

# Print predicted vs actual values for the first 10 test samples
pred_val = sgd_pipeline.predict(X_test)

print('Predicted Value:', pred_val[:10])
print()
print('Actual Value:', y_test[:10])

SGD Output: {'model': 'SGD Regressor', 'Train RMSE': np.float64(29223.6609), 'Test RMSE': np.float64(29144.5935), 'Train R2': 0.9867, 'Test R2': 0.9868}

Predicted Value: [ 850586.69792271  508767.88068632  984674.98436705 1023063.59636492
  761591.50530244  766351.7537549  1000584.44960717  899802.04725083
  797701.12905227  888849.07454817]

Actual Value: 521    9.010005e+05
737    4.945375e+05
740    9.494042e+05
660    1.040389e+06
411    7.940100e+05
678    7.240336e+05
626    9.984392e+05
513    9.097134e+05
859    7.926815e+05
136    9.474908e+05
Name: House_Price, dtype: float64


---
## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:
- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [ ]:
# Question 5
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score

X = clean_df.drop('House_Price', axis = 1)
y = clean_df['House_Price']

X_train, X_test , y_train, y_test = train_test_split(X,y, test_size = 0.20 , random_state = 42)

# LinearRegression as the final estimator
# The ColumnTransformer
lin_pipeline = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', LinearRegression())
    ]
)

# Train the pipeline and evaluate using RMSE and R² score
# Train the pipeline
train_lin = lin_pipeline.fit(X_train, y_train)

# evaluate using RMSE and R² score
def lin_evaluate(model, name):
  train_pred = model.predict(X_train)
  test_pred = model.predict(X_test)

  train_RMSE = np.sqrt(mean_squared_error(y_train, train_pred))
  test_RMSE = np.sqrt(mean_squared_error(y_test, test_pred))
  train_R2 = r2_score(y_train, train_pred)
  test_R2 = r2_score(y_test, test_pred)

  return {
      'model'             : name,
      'Train RMSE'        : round(train_RMSE, 4),
      'Test RMSE'         : round(test_RMSE, 4),
      'Train R2'          : round(train_R2, 4),
      'Test R2'           : round(test_R2, 4)
  }

# LinearRegression as the final estimator
lin_result = lin_evaluate(lin_pipeline, 'Linear Regression')
print('Linear Output:', lin_result)
print()

# Print predicted vs actual values for the first 10 test samples
pred_val = lin_pipeline.predict(X_test)
print('Predicted Value: \n', pred_val[:10])
print()
print('Actual Value: \n', y_test[:10])

Linear Output: {'model': 'Linear Regression', 'Train RMSE': np.float64(29142.5234), 'Test RMSE': np.float64(29093.0752), 'Train R2': 0.9868, 'Test R2': 0.9869}

Predicted Value: 
 [ 851409.14732655  507641.0338697   985990.37503164 1022654.70653384
  760698.13365681  765782.31638091 1005039.09634383  903407.78177236
  797971.18840959  888850.4300377 ]

Actual Value: 
 521    9.010005e+05
737    4.945375e+05
740    9.494042e+05
660    1.040389e+06
411    7.940100e+05
678    7.240336e+05
626    9.984392e+05
513    9.097134e+05
859    7.926815e+05
136    9.474908e+05
Name: House_Price, dtype: float64


---
## Question 6 (10 Marks) (You have to explore the topic and use the equation via Numpy)
### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:
- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)

In [ ]:
# Question 7
# Pick random 100 datas from the dataset
df_random = clean_df.sample(n = 100, random_state=42)

# Use Square_Footage, Num_Bedrooms, and Num_Bathrooms as features
X = df_random[['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']]
y = df_random['House_Price']

one = np.ones((len(X), 1))
X = np.append(one, X, axis = 1)
y = np.array(y).reshape((len(y), 1))

# Use the Normal Equation: θ = (XᵀX)⁻¹ Xᵀy
theta = np.dot((np.linalg.inv(np.dot(X.T, X))), np.dot(X.T, y))

# Print the learned coefficients (θ values)
print('Intercept:', theta[0])
print('Square Footage:', theta[1])
print('Num Bedrooms:', theta[2])
print('Num Bathrooms:', theta[3])

Intercept: [20888.25945218]
Square Footage: [202.30820792]
Num Bedrooms: [8303.2810412]
Num Bathrooms: [3020.32852173]
